In [ ]:
import json
from spark_session_config import spark
from pyspark.sql import functions as F
from pyspark.sql import types as T
from kafka import KafkaProducer
from schema import sensor_data_schema

models_df = spark.read.parquet("s3a://pyspark/data/dims/car_models")
colors_df = spark.read.parquet("s3a://pyspark/data/dims/car_colors")
cars_df = spark.read.parquet("s3a://pyspark/data/dims/cars")

models_df.cache()
colors_df.cache()
cars_df.cache()

stream_df = spark.readStream.format("kafka") \
.option("kafka.bootstrap.servers", "course-kafka:9092") \
.option("subscribe", "sensors-sample") \
.option("startingOffsets", "earliest") \
.load() \
.select(F.col("value").cast("string"))


parsed_df = stream_df \
.withColumn('parsed_json', F.from_json(F.col("value"), sensor_data_schema)) \
.select(F.col('parsed_json.*'))

cars_data_df = parsed_df \
            .join(cars_df,on = 'car_id', how = 'left') \
            .join(models_df,on = 'model_id', how = 'left') \
            .join(colors_df,on = 'color_id', how='left')

enriched_df = cars_data_df \
    .select(     
        F.col("event_id"),   
        F.col("event_time"),   
        F.col("car_id"),   
        F.col("speed"),   
        F.col("rpm"),  
        F.col("gear"),
        F.col("driver_id"),            
        F.col("car_brand").alias("brand_name"),
        F.col("car_model").alias("model_name"),
        F.col("color_name"),
       (F.round(F.col("speed") / F.lit(30))).cast("int").alias("expected_gear")
    )

data_json_df = enriched_df.select(
    F.to_json(F.struct("*")).alias("value")
)

# send to new topic

data_json_df \
.writeStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "course-kafka:9092") \
    .option("topic", "samples-enriched") \
    .option("checkpointLocation", 's3a://pyspark/checkpoints/checkpoints/enrichment') \
    .outputMode("update") \
    .start() \
    .awaitTermination()

spark.stop()


26/08/16 23:38:57 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


DataFrame[car_id: bigint, driver_id: int, model_id: int, color_id: int]